MINI ASSIGNMENT

In [1]:
!pip install -q gitpython transformers accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.5/228.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 2.3 MB/s eta 0:00:00


In [1]:
import torch

print("PyTorch version:", torch.__version__)

if torch.cuda.is_available():
    print("GPU available:", torch.cuda.get_device_name(0))
else:
    print("GPU not available - using CPU")

PyTorch version: 2.11.0+cu130
GPU available: Tesla T4


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Qwen model loaded successfully!")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen model loaded successfully!


In [3]:
prompt = """
Explain this simple Python code in very simple language.
Mention what the code does and how it works.

Code:

def add(a, b):
    return a + b

result = add(5, 3)
print(result)
"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

This Python code defines a function called `add` that takes two arguments, `a` and `b`, and returns their sum. The function is then called with the arguments 5 and 3, and the result of adding these numbers is stored in the variable `result`. Finally, the result is printed to the console using the `print` statement.


In [5]:
!pip install -q GitPython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.5/228.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 3.1 MB/s eta 0:00:00


In [6]:
from git import Repo

print("GitPython installed successfully!")

GitPython installed successfully!


In [7]:
import os
import shutil
from git import Repo

github_url = input("Enter GitHub Repository URL: ")

repo_folder = "/content/github_repo"

if os.path.exists(repo_folder):
    shutil.rmtree(repo_folder)

Repo.clone_from(github_url, repo_folder)

print("\nRepository cloned successfully!")
print("Repository location:", repo_folder)

Enter GitHub Repository URL: https://github.com/Chaturyathukivakam/GenAI

Repository cloned successfully!
Repository location: /content/github_repo


In [9]:
import os
import json

def extract_repository_code(repo_folder):
    all_code = []

    for root, dirs, files in os.walk(repo_folder):

        # Ignore unnecessary folders
        dirs[:] = [
            d for d in dirs
            if d not in [".git", "__pycache__", "venv", ".venv"]
        ]

        for file in files:

            file_path = os.path.join(root, file)

            # Python files
            if file.endswith(".py"):

                try:
                    with open(
                        file_path,
                        "r",
                        encoding="utf-8",
                        errors="ignore"
                    ) as f:

                        content = f.read()

                    all_code.append(
                        f"\n===== FILE: {file} =====\n"
                        f"{content}"
                    )

                except Exception:
                    pass

            # Jupyter notebooks
            elif file.endswith(".ipynb"):

                try:
                    with open(
                        file_path,
                        "r",
                        encoding="utf-8"
                    ) as f:

                        notebook = json.load(f)

                    notebook_code = []

                    for cell in notebook.get("cells", []):

                        if cell.get("cell_type") == "code":

                            source = "".join(
                                cell.get("source", [])
                            )

                            notebook_code.append(source)

                    if notebook_code:

                        all_code.append(
                            f"\n===== NOTEBOOK: {file} =====\n"
                            + "\n".join(notebook_code)
                        )

                except Exception:
                    pass

    return "\n".join(all_code)


repository_code = extract_repository_code(repo_folder)

print("Code extraction completed!")
print("Total characters extracted:", len(repository_code))

Code extraction completed!
Total characters extracted: 15032


In [10]:
# Limit the code so the model can process it comfortably
code_for_model = repository_code[:12000]

prompt = f"""
You are a software project analyst.

Analyze the following code extracted from a GitHub repository.

Give a simple and accurate explanation that a college student
can understand.

Use the following format:

# Project Overview
Explain what the project does.

# Main Technologies
List the programming languages, libraries, frameworks,
and tools that can be identified from the code.

# Main Features
List the important features or tasks implemented.

# How It Works
Explain the overall workflow of the project step by step.

# Important Files
Mention the important notebooks or source files and
briefly explain what each one contains.

Do not invent information.
Only use information that can be understood from the
provided repository code.

Repository Code:
{code_for_model}
"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=8192
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=700,
    do_sample=False
)

explanation = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(explanation)

### Project Overview

This project involves creating a deep learning model for generating synthetic images based on pre-trained models like VGG19. The primary goal is to create an image synthesis system capable of producing high-quality images similar to fashion images.

### Main Technologies

#### Python Libraries
- **PyTorch**: Used for implementing the neural network architecture.
- **Ninja**: A tool for managing dependencies between different parts of the project.
- **matplotlib**: For plotting the generated images.

#### Other Libraries
- **PIL**: For handling image operations.
- **OpenCV**: For image processing.
- **requests**: For making HTTP requests to fetch image data.

#### Frameworks
- **TensorFlow/Keras**: For building the neural network model.
- **Scikit-Learn**: For machine learning tasks like logistic regression.

### Main Features

#### Model Architecture
The model consists of two main components:
1. **Generator**:
   - Uses a VGG19 architecture to generate synthetic i

In [11]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 84.0 MB/s eta 0:00:00


In [12]:
%%writefile app.py

import streamlit as st
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import os
import json
import shutil
import tempfile
from git import Repo


# -----------------------------
# Load Local Qwen Model
# -----------------------------

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

@st.cache_resource
def load_model():

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=torch.float16,
        device_map="auto"
    )

    return tokenizer, model


tokenizer, model = load_model()


# -----------------------------
# Extract Repository Code
# -----------------------------

def extract_repository_code(repo_folder):

    all_code = []

    for root, dirs, files in os.walk(repo_folder):

        dirs[:] = [
            d for d in dirs
            if d not in [
                ".git",
                "__pycache__",
                "venv",
                ".venv"
            ]
        ]

        for file in files:

            file_path = os.path.join(root, file)

            # Python files
            if file.endswith(".py"):

                try:

                    with open(
                        file_path,
                        "r",
                        encoding="utf-8",
                        errors="ignore"
                    ) as f:

                        content = f.read()

                    all_code.append(
                        f"\n===== FILE: {file} =====\n"
                        f"{content}"
                    )

                except Exception:
                    pass

            # Jupyter notebooks
            elif file.endswith(".ipynb"):

                try:

                    with open(
                        file_path,
                        "r",
                        encoding="utf-8"
                    ) as f:

                        notebook = json.load(f)

                    notebook_code = []

                    for cell in notebook.get("cells", []):

                        if cell.get("cell_type") == "code":

                            source = "".join(
                                cell.get("source", [])
                            )

                            notebook_code.append(source)

                    if notebook_code:

                        all_code.append(
                            f"\n===== NOTEBOOK: {file} =====\n"
                            + "\n".join(notebook_code)
                        )

                except Exception:
                    pass

    return "\n".join(all_code)


# -----------------------------
# Generate Explanation
# -----------------------------

def generate_explanation(code):

    code = code[:12000]

    prompt = f"""
You are a GitHub repository code explainer.

Analyze ONLY the repository code provided below.

Do not guess or invent technologies, features,
files, models, or functionality.

If something cannot be determined from the code,
say "Not clearly identified from the available code."

Give the answer in simple language.

Use this format:

PROJECT OVERVIEW
Explain what the repository does.

MAIN TECHNOLOGIES
List only technologies clearly visible in the code.

MAIN FEATURES
List important features clearly visible in the code.

HOW IT WORKS
Explain the workflow step by step.

IMPORTANT FILES
Mention important files and explain their purpose.

Repository code:

{code}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=8192
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=600,
        do_sample=False
    )

    explanation = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return explanation


# -----------------------------
# Streamlit Interface
# -----------------------------

st.set_page_config(
    page_title="GitHub Code Explainer",
    page_icon="💻"
)

st.title("💻 GitHub Repository Code Explainer")

st.write(
    "Enter a public GitHub repository URL "
    "to generate a simple explanation of the codebase."
)

github_url = st.text_input(
    "GitHub Repository URL",
    placeholder="https://github.com/username/repository"
)


if st.button("Explain Repository"):

    if not github_url:

        st.warning(
            "Please enter a GitHub repository URL."
        )

    else:

        try:

            with st.spinner(
                "Cloning repository..."
            ):

                repo_folder = tempfile.mkdtemp()

                Repo.clone_from(
                    github_url,
                    repo_folder
                )

            with st.spinner(
                "Extracting source code..."
            ):

                repository_code = (
                    extract_repository_code(
                        repo_folder
                    )
                )

            shutil.rmtree(
                repo_folder,
                ignore_errors=True
            )

            if not repository_code:

                st.error(
                    "No supported source-code files were found."
                )

            else:

                st.info(
                    f"Extracted {len(repository_code):,} characters of code."
                )

                with st.spinner(
                    "Qwen is analyzing the repository..."
                ):

                    explanation = (
                        generate_explanation(
                            repository_code
                        )
                    )

                st.success(
                    "Repository analyzed successfully!"
                )

                st.markdown(explanation)

        except Exception as e:

            st.error(
                f"Error: {e}"
            )

Writing app.py


In [13]:
!streamlit run app.py &>/content/streamlit.log &

In [14]:
!pip install -q pyngrok

In [17]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 22 packages in 4s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸

In [5]:
!streamlit run app.py --server.port 8501 > /content/streamlit.log 2>&1 &

In [6]:
!cat /content/streamlit.log

/bin/bash: line 1: streamlit: command not found


In [7]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 78.2 MB/s eta 0:00:00


In [8]:
!streamlit run app.py --server.port 8501 > /content/streamlit.log 2>&1 &

In [9]:
!cat /content/streamlit.log

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


In [10]:
!find /content -name "app.py" -type f

In [12]:
import os

print("Repository exists:", os.path.exists("/content/github_repo"))
print("App exists:", os.path.exists("/content/app.py"))

Repository exists: False
App exists: False


In [13]:
!pip install -q GitPython transformers accelerate sentencepiece streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.5/228.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 4.8 MB/s eta 0:00:00


In [14]:
import os
import shutil
from git import Repo

github_url = "https://github.com/Chaturyathukivakam/GenAI"
repo_folder = "/content/github_repo"

if os.path.exists(repo_folder):
    shutil.rmtree(repo_folder)

Repo.clone_from(github_url, repo_folder)

print("Repository cloned successfully!")
print("Repository location:", repo_folder)

Repository cloned successfully!
Repository location: /content/github_repo


In [15]:
%%writefile /content/app.py

import os
import json
import shutil
import streamlit as st
import torch

from git import Repo
from transformers import AutoTokenizer, AutoModelForCausalLM


# -----------------------------
# Configuration
# -----------------------------

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"


# -----------------------------
# Load Qwen Model
# -----------------------------

@st.cache_resource
def load_model():

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    return tokenizer, model


# -----------------------------
# Clone GitHub Repository
# -----------------------------

def clone_repository(github_url):

    repo_folder = "/content/github_repo"

    if os.path.exists(repo_folder):
        shutil.rmtree(repo_folder)

    Repo.clone_from(github_url, repo_folder)

    return repo_folder


# -----------------------------
# Extract Code
# -----------------------------

def extract_repository_code(repo_folder):

    all_code = []

    for root, dirs, files in os.walk(repo_folder):

        dirs[:] = [
            d for d in dirs
            if d not in [".git", "__pycache__", "venv", ".venv"]
        ]

        for file in files:

            file_path = os.path.join(root, file)

            # Python files
            if file.endswith(".py"):

                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        code = f.read()

                    all_code.append(
                        f"\n\n===== FILE: {file} =====\n{code}"
                    )

                except Exception:
                    pass

            # Jupyter notebooks
            elif file.endswith(".ipynb"):

                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        notebook = json.load(f)

                    for cell in notebook.get("cells", []):

                        if cell.get("cell_type") == "code":

                            code = "".join(cell.get("source", []))

                            all_code.append(
                                f"\n\n===== FILE: {file} =====\n{code}"
                            )

                except Exception:
                    pass

    return "\n".join(all_code)


# -----------------------------
# Generate Explanation
# -----------------------------

def generate_explanation(code, tokenizer, model):

    code = code[:12000]

    prompt = f"""
You are a code explanation assistant.

Analyze ONLY the code provided below.

Do not invent files, technologies, features, libraries, models,
functions, or results that are not clearly present in the code.

Give a simple explanation using these sections:

1. Project Overview
2. Main Technologies
3. Main Features
4. How It Works
5. Important Files

If something cannot be determined from the code, say:
"Not clearly identified in the provided code."

Repository Code:

{code}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=8192
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=600,
            do_sample=False
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    explanation = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return explanation


# -----------------------------
# Streamlit Interface
# -----------------------------

st.set_page_config(
    page_title="GitHub Repository Code Explainer",
    page_icon="💻"
)

st.title("💻 Local GitHub Repository Code Explainer")

st.write(
    "Enter a public GitHub repository URL to automatically "
    "analyze and explain its source code."
)

github_url = st.text_input(
    "GitHub Repository URL",
    placeholder="https://github.com/username/repository"
)


if st.button("Explain Repository"):

    if not github_url:

        st.warning("Please enter a GitHub repository URL.")

    else:

        try:

            with st.spinner("Cloning repository..."):

                repo_folder = clone_repository(github_url)

            st.success("Repository cloned successfully!")

            with st.spinner("Extracting source code..."):

                repository_code = extract_repository_code(repo_folder)

            if not repository_code:

                st.error(
                    "No Python (.py) or Jupyter Notebook (.ipynb) "
                    "code was found."
                )

            else:

                st.info(
                    f"Code extracted: {len(repository_code)} characters"
                )

                with st.spinner("Loading Qwen model..."):

                    tokenizer, model = load_model()

                with st.spinner("Generating explanation..."):

                    explanation = generate_explanation(
                        repository_code,
                        tokenizer,
                        model
                    )

                st.subheader("📖 Repository Explanation")

                st.markdown(explanation)

        except Exception as e:

            st.error(f"Error: {str(e)}")

Writing /content/app.py


In [16]:
!streamlit run /content/app.py --server.port 8501 > /content/streamlit.log 2>&1 &

In [17]:
!cat /content/streamlit.log



2026-10-05 16:39:29.275 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.168.203.95:8501



In [18]:
!lt --port 8501

/bin/bash: line 1: lt: command not found


In [20]:
!curl -I http://localhost:8501

HTTP/1.1 200 OK
date: Mon, 05 Oct 2026 16:42:03 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 6463
last-modified: Mon, 05 Oct 2026 16:30:22 GMT
etag: "7add5feb2b48543838725ffb1b400dad"
cache-control: no-cache



In [22]:
from google.colab import output

output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [23]:
import sys
sys.path.append("/content")

from app import extract_repository_code, clone_repository

repo_folder = "/content/github_repo"

code = extract_repository_code(repo_folder)

print("Repository found:", True)
print("Code extracted:", len(code), "characters")
print("\nFirst 500 characters:\n")
print(code[:500])

2026-10-05 16:46:24.752 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 16:46:24.753 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 16:46:24.890 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-10-05 16:46:24.891 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 16:46:24.893 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 16:46:24.894 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 16:46:24.895 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

Repository found: True
Code extracted: 16946 characters

First 500 characters:



===== FILE: ASSIGNMENT_4.ipynb =====
# Install PyTorch if required
!pip install torch torchvision matplotlib -q

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os


===== FILE: ASSIGNMENT_4.ipynb =====
# Check whether GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")




In [24]:
from app import load_model, generate_explanation

print("Loading Qwen model...")

tokenizer, model = load_model()

print("Qwen loaded successfully!")

explanation = generate_explanation(
    code,
    tokenizer,
    model
)

print("\n===== GENERATED EXPLANATION =====\n")
print(explanation)

2026-10-05 16:47:37.021 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Loading Qwen model...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen loaded successfully!

===== GENERATED EXPLANATION =====

This notebook provides an overview of the project structure and its components. Here's a breakdown of each section:

### Project Overview
The project is titled "Assignment 4," which involves creating a neural style transfer model using PyTorch. The main focus is on generating synthetic images based on pre-trained styles learned from fashion MNIST data.

### Main Technologies
- **PyTorch**: The primary framework used for deep learning tasks.
- **Transformers**: For handling image transformations like resizing and normalization.
- **Data Loading**: Utilizing `torchvision` for loading and preprocessing image data.
- **Generative Adversarial Networks (GANs)**: Used for generating new images based on existing ones.
- **Style Transfer Models**: Implemented using VGG19 architecture for style transfer.

### Main Features
- **Generation of Synthetic Images**: Generates new images based on pre-trained styles learned from fashion MNIST

In [25]:
!pip install -q fastapi uvicorn pydantic

In [26]:
%%writefile /content/backend.py

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import os
import shutil
import json
from git import Repo

from app import load_model, generate_explanation


app = FastAPI(
    title="GitHub Repository Code Explainer API",
    description="API for analyzing and explaining GitHub repository code",
    version="1.0"
)


class RepositoryRequest(BaseModel):
    github_url: str


def clone_repository(github_url):

    repo_folder = "/content/api_github_repo"

    if os.path.exists(repo_folder):
        shutil.rmtree(repo_folder)

    Repo.clone_from(github_url, repo_folder)

    return repo_folder


def extract_repository_code(repo_folder):

    all_code = []

    for root, dirs, files in os.walk(repo_folder):

        dirs[:] = [
            d for d in dirs
            if d not in [".git", "__pycache__", "venv", ".venv"]
        ]

        for file in files:

            file_path = os.path.join(root, file)

            if file.endswith(".py"):

                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        code = f.read()

                    all_code.append(
                        f"\n===== FILE: {file} =====\n{code}"
                    )

                except Exception:
                    pass

            elif file.endswith(".ipynb"):

                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        notebook = json.load(f)

                    for cell in notebook.get("cells", []):

                        if cell.get("cell_type") == "code":

                            code = "".join(
                                cell.get("source", [])
                            )

                            all_code.append(
                                f"\n===== FILE: {file} =====\n{code}"
                            )

                except Exception:
                    pass

    return "\n".join(all_code)


@app.get("/")
def home():

    return {
        "message": "GitHub Repository Code Explainer API is running"
    }


@app.post("/explain")
def explain_repository(request: RepositoryRequest):

    try:

        # Clone repository
        repo_folder = clone_repository(request.github_url)

        # Extract code
        repository_code = extract_repository_code(repo_folder)

        if not repository_code:

            raise HTTPException(
                status_code=400,
                detail="No Python or Jupyter Notebook code found."
            )

        # Load Qwen
        tokenizer, model = load_model()

        # Generate explanation
        explanation = generate_explanation(
            repository_code,
            tokenizer,
            model
        )

        return {
            "github_url": request.github_url,
            "characters_extracted": len(repository_code),
            "explanation": explanation
        }

    except HTTPException:
        raise

    except Exception as e:

        raise HTTPException(
            status_code=500,
            detail=str(e)
        )

Writing /content/backend.py


In [27]:
!uvicorn backend:app --host 0.0.0.0 --port 8000 > /content/fastapi.log 2>&1 &

In [29]:
!cat /content/fastapi.log

2026-10-05 16:53:55.331 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 16:53:55.331 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 16:53:55.586 
  command:

    streamlit run /usr/local/bin/uvicorn [ARGUMENTS]
2026-10-05 16:53:55.586 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 16:53:55.587 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 16:53:55.587 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 16:53:55.587 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 16:53:55.587 

In [30]:
!ps aux | grep uvicorn

root        7808 19.9  6.3 4301232 840788 ?      Sl   16:53   0:11 /usr/bin/python3 /usr/local/bin/uvicorn backend:app --host 0.0.0.0 --port 8000
root        8071  0.0  0.0   7348  3512 ?        S    16:54   0:00 /bin/bash -c ps aux | grep uvicorn
root        8073  0.0  0.0   6552  2512 ?        S    16:54   0:00 grep uvicorn


In [31]:
import requests

response = requests.get("http://localhost:8000/")

print("Status:", response.status_code)
print("Response:", response.json())

Status: 200
Response: {'message': 'GitHub Repository Code Explainer API is running'}


In [32]:
import requests

url = "http://localhost:8000/explain"

data = {
    "github_url": "https://github.com/Chaturyathukivakam/GenAI"
}

response = requests.post(url, json=data)

print("Status:", response.status_code)
print(response.json())

Status: 200
{'github_url': 'https://github.com/Chaturyathukivakam/GenAI', 'characters_extracted': 16892, 'explanation': "The provided code snippet appears to be an extensive notebook containing multiple IPython notebooks, each focusing on different aspects of generating artificial images based on specific styles. Here's a simplified breakdown of what each section does:\n\n### Project Overview\nThis project involves creating a neural network model called `StyleGAN2` that can generate synthetic images based on given styles. The main components include:\n- **Generative Adversarial Networks (GANs)**: Used to create the style transfer process.\n- **Transfer Learning**: Loading pre-trained models like VGG19 to improve performance.\n- **Data Augmentation**: Applying transformations to enhance the quality of the generated images.\n- **Style Transfer**: Using the pre-trained model to apply various styles to synthetic images.\n\n### Main Technologies\n- **PyTorch**: The primary framework used fo

In [33]:
%%writefile /tmp/new_function.txt
def generate_explanation(code, tokenizer, model):

    code = code[:12000]

    prompt = f"""
You are a software code analysis assistant.

IMPORTANT RULES:
1. Analyze ONLY the code provided below.
2. Do NOT guess or invent technologies, models, files, features,
   datasets, algorithms, or results.
3. Mention a technology ONLY if it is clearly visible in the code,
   such as an import, class, function, or explicit configuration.
4. Do NOT assume that an imported library was actually used.
5. Do NOT describe functionality that is not visible in the code.
6. If information is uncertain or missing, write:
   "Not clearly identified in the provided code."
7. Keep the explanation simple and suitable for a college project.
8. Use the actual filenames shown in the code.
9. Do not confuse library names with the purpose of the project.

Return exactly these sections:

### Project Overview
Briefly describe what the repository appears to do based only on the code.

### Main Technologies
List only technologies/libraries clearly present in the code.

### Main Features
List only features directly supported by the code.

### How It Works
Explain the main processing flow visible in the code.

### Important Files
List the actual files visible in the provided repository code.

Repository Code:
{code}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=8192
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=600,
            do_sample=False
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    explanation = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return explanation

Writing /tmp/new_function.txt


In [34]:
# Show the current app.py so we can verify the function location
!grep -n "def generate_explanation" /content/app.py

113:def generate_explanation(code, tokenizer, model):


In [35]:
%%writefile /content/app.py

import os
import json
import shutil
import streamlit as st
import torch

from git import Repo
from transformers import AutoTokenizer, AutoModelForCausalLM


MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"


@st.cache_resource
def load_model():

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=torch.float16,
        device_map="auto"
    )

    return tokenizer, model


def clone_repository(github_url):

    repo_folder = "/content/github_repo"

    if os.path.exists(repo_folder):
        shutil.rmtree(repo_folder)

    Repo.clone_from(github_url, repo_folder)

    return repo_folder


def extract_repository_code(repo_folder):

    all_code = []

    for root, dirs, files in os.walk(repo_folder):

        dirs[:] = [
            d for d in dirs
            if d not in [".git", "__pycache__", "venv", ".venv"]
        ]

        for file in files:

            file_path = os.path.join(root, file)

            if file.endswith(".py"):

                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        code = f.read()

                    all_code.append(
                        f"\n===== FILE: {file} =====\n{code}"
                    )

                except Exception:
                    pass

            elif file.endswith(".ipynb"):

                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        notebook = json.load(f)

                    for cell in notebook.get("cells", []):

                        if cell.get("cell_type") == "code":

                            code = "".join(
                                cell.get("source", [])
                            )

                            all_code.append(
                                f"\n===== FILE: {file} =====\n{code}"
                            )

                except Exception:
                    pass

    return "\n".join(all_code)


def generate_explanation(code, tokenizer, model):

    code = code[:12000]

    prompt = f"""
You are a software code analysis assistant.

IMPORTANT RULES:

1. Analyze ONLY the code provided below.
2. Do NOT guess or invent technologies, models, files,
   features, datasets, algorithms, or results.
3. Mention a technology ONLY if it is clearly visible in the code.
4. Do NOT assume an imported library was actually used.
5. Do NOT describe functionality that is not visible in the code.
6. If information is uncertain or missing, write:
   "Not clearly identified in the provided code."
7. Keep the explanation simple and suitable for a college project.
8. Use the actual filenames shown in the code.
9. Do not confuse library names with the purpose of the project.

Return exactly these sections:

### Project Overview
Briefly describe what the repository appears to do based only on the code.

### Main Technologies
List only technologies/libraries clearly present in the code.

### Main Features
List only features directly supported by the code.

### How It Works
Explain the main processing flow visible in the code.

### Important Files
List the actual files visible in the provided repository code.

Repository Code:

{code}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=8192
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=600,
            do_sample=False
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    explanation = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return explanation


st.set_page_config(
    page_title="GitHub Repository Code Explainer",
    page_icon="💻"
)

st.title("💻 Local GitHub Repository Code Explainer")

st.write(
    "Enter a public GitHub repository URL to automatically "
    "analyze and explain its source code."
)

github_url = st.text_input(
    "GitHub Repository URL",
    placeholder="https://github.com/username/repository"
)


if st.button("Explain Repository"):

    if not github_url:

        st.warning("Please enter a GitHub repository URL.")

    else:

        try:

            with st.spinner("Cloning repository..."):

                repo_folder = clone_repository(github_url)

            st.success("Repository cloned successfully!")

            with st.spinner("Extracting source code..."):

                repository_code = extract_repository_code(repo_folder)

            if not repository_code:

                st.error(
                    "No Python (.py) or Jupyter Notebook (.ipynb) "
                    "code was found."
                )

            else:

                st.info(
                    f"Code extracted: {len(repository_code)} characters"
                )

                with st.spinner("Loading Qwen model..."):

                    tokenizer, model = load_model()

                with st.spinner("Generating explanation..."):

                    explanation = generate_explanation(
                        repository_code,
                        tokenizer,
                        model
                    )

                st.subheader("📖 Repository Explanation")

                st.markdown(explanation)

        except Exception as e:

            st.error(f"Error: {str(e)}")

Overwriting /content/app.py


In [36]:
!pkill -f "uvicorn backend:app" || true

^C


In [37]:
!uvicorn backend:app --host 0.0.0.0 --port 8000 > /content/fastapi.log 2>&1 &

In [38]:
!cat /content/fastapi.log

In [39]:
!cat /content/fastapi.log


2026-10-05 17:00:49.870 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 17:00:49.871 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 17:00:50.095 
  command:

    streamlit run /usr/local/bin/uvicorn [ARGUMENTS]
2026-10-05 17:00:50.095 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 17:00:50.095 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 17:00:50.096 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 17:00:50.096 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 17:00:50.096 

In [40]:
import requests

response = requests.post(
    "http://localhost:8000/explain",
    json={
        "github_url": "https://github.com/Chaturyathukivakam/GenAI"
    }
)

print("Status:", response.status_code)
print(response.json()["explanation"])

Status: 200
The provided code snippet appears to be a Python script designed to perform various tasks related to generating synthetic images using CycleGAN architecture from the Pix2Pix model. Here's a breakdown of the key components and functionalities within the script:

### Project Overview
The repository appears to be focused on implementing CycleGAN, which is a generative adversarial network (GAN) framework for creating high-quality synthetic images. The primary goal seems to be generating images that closely resemble the original input while being altered through a cycle process.

### Main Technologies
The script uses several libraries and modules to achieve its goals:

1. **PyTorch**: For deep learning operations such as neural networks, data loading, and model training.
2. **Transformers**: For handling image transformations like resizing and normalization.
3. **Libraries**: `matplotlib` for plotting images, `PIL` for image manipulation, and `requests` for fetching URLs.
4. **L

In [41]:
# Check exactly which files are being extracted

import os

repo_folder = "/content/github_repo"

for root, dirs, files in os.walk(repo_folder):
    dirs[:] = [d for d in dirs if d not in [".git", "__pycache__", "venv", ".venv"]]

    for file in files:
        if file.endswith(".py") or file.endswith(".ipynb"):
            print(os.path.join(root, file))

/content/github_repo/ASSIGNMENT_4.ipynb
/content/github_repo/ASSIGNMENT 5 GEN AI .ipynb
/content/github_repo/ASSIGNMENT_6.ipynb
/content/github_repo/Untitled11.ipynb
/content/github_repo/Untitled13.ipynb


In [42]:
import os
import json

repo_folder = "/content/github_repo"

for file in os.listdir(repo_folder):
    if file.endswith(".ipynb"):
        path = os.path.join(repo_folder, file)

        with open(path, "r", encoding="utf-8") as f:
            notebook = json.load(f)

        code_cells = [
            cell for cell in notebook.get("cells", [])
            if cell.get("cell_type") == "code"
        ]

        total_chars = sum(
            len("".join(cell.get("source", [])))
            for cell in code_cells
        )

        print(f"\n📄 {file}")
        print(f"   Code cells: {len(code_cells)}")
        print(f"   Code characters: {total_chars}")



📄 ASSIGNMENT_4.ipynb
   Code cells: 16
   Code characters: 7133

📄 ASSIGNMENT 5 GEN AI .ipynb
   Code cells: 8
   Code characters: 1917

📄 ASSIGNMENT_6.ipynb
   Code cells: 9
   Code characters: 1721

📄 Untitled11.ipynb
   Code cells: 6
   Code characters: 1605

📄 Untitled13.ipynb
   Code cells: 15
   Code characters: 2389


In [43]:
import json
import os

repo_folder = "/content/github_repo"

for file in os.listdir(repo_folder):
    if file.endswith(".ipynb"):
        path = os.path.join(repo_folder, file)

        with open(path, "r", encoding="utf-8") as f:
            notebook = json.load(f)

        code = ""

        for cell in notebook.get("cells", []):
            if cell.get("cell_type") == "code":
                code += "".join(cell.get("source", [])) + "\n"

        print("\n" + "=" * 60)
        print("FILE:", file)
        print("=" * 60)
        print(code[:1000])


FILE: ASSIGNMENT_4.ipynb
# Install PyTorch if required
!pip install torch torchvision matplotlib -q

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os
# Check whether GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# Parameters
batch_size = 128
image_size = 64
latent_dim = 100
num_epochs = 30
learning_rate = 0.0002
beta1 = 0.5

# Create folder to save generated images
os.makedirs("generated_images", exist_ok=True)
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

dataloader = DataLoader(
    dataset,
    batch_size=batch_size

In [44]:
# Test Qwen on ONE actual notebook

import json
import os
import torch

file_path = "/content/github_repo/ASSIGNMENT_4.ipynb"

with open(file_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

code = ""

for cell in notebook.get("cells", []):
    if cell.get("cell_type") == "code":
        code += "".join(cell.get("source", [])) + "\n"

code = code[:7000]

prompt = f"""
You are analyzing a Jupyter Notebook for a college project.

Analyze ONLY the code below.

IMPORTANT:
- Do not guess.
- Do not invent technologies or algorithms.
- Identify the actual dataset, libraries, model architecture,
  and purpose only when clearly visible.
- Ignore comments that contradict the actual code.
- Keep the answer short and factual.

Notebook filename:
ASSIGNMENT_4.ipynb

Return exactly:

Project:
Dataset:
Main Libraries:
Model:
Main Process:
Output:

CODE:
{code}
"""

messages = [{"role": "user", "content": prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=8192
)

inputs = {
    k: v.to(model.device)
    for k, v in inputs.items()
}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False
    )

generated = outputs[0][inputs["input_ids"].shape[1]:]

print(tokenizer.decode(
    generated,
    skip_special_tokens=True
))

Project: Assignment_4
Dataset: None
Main Libraries: torch, torchvision, matplotlib
Model: None
Main Process: 
Output: 
None


In [45]:
import re

code = """
import torch
import torch.nn as nn
from torchvision import datasets
dataset = datasets.FashionMNIST(...)
class Generator(nn.Module):
    pass
class Discriminator(nn.Module):
    pass
"""

libraries = re.findall(
    r'(?:import|from)\s+([A-Za-z0-9_]+)',
    code
)

keywords = [
    "FashionMNIST",
    "MNIST",
    "Generator",
    "Discriminator",
    "VGG19",
    "LogisticRegression",
    "gpt2",
    "pipeline"
]

found = [word for word in keywords if word in code]

print("Libraries:", list(set(libraries)))
print("Detected:", found)

Libraries: ['datasets', 'torch', 'torchvision']
Detected: ['FashionMNIST', 'MNIST', 'Generator', 'Discriminator']


In [46]:
import os
import json
import re

repo_folder = "/content/github_repo"

patterns = {
    "Datasets": [
        "FashionMNIST",
        "MNIST",
        "CIFAR10",
        "CIFAR100",
        "ImageNet"
    ],
    "Models": [
        "Generator",
        "Discriminator",
        "VGG19",
        "LogisticRegression",
        "gpt2",
        "GPT2"
    ],
    "Libraries": [
        "torch",
        "torchvision",
        "tensorflow",
        "transformers",
        "numpy",
        "matplotlib",
        "PIL",
        "sklearn",
        "requests",
        "pandas"
    ]
}

for filename in os.listdir(repo_folder):

    if not filename.endswith(".ipynb"):
        continue

    path = os.path.join(repo_folder, filename)

    with open(path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    code = ""

    for cell in notebook.get("cells", []):
        if cell.get("cell_type") == "code":
            code += "".join(cell.get("source", [])) + "\n"

    print("\n" + "=" * 60)
    print("FILE:", filename)
    print("=" * 60)

    for category, items in patterns.items():

        found = []

        for item in items:
            # Exact word matching
            if re.search(r"\b" + re.escape(item) + r"\b", code):
                found.append(item)

        if found:
            print(category + ":", ", ".join(sorted(set(found))))


FILE: ASSIGNMENT_4.ipynb
Datasets: FashionMNIST, MNIST
Models: Discriminator, Generator
Libraries: matplotlib, torch, torchvision

FILE: ASSIGNMENT 5 GEN AI .ipynb
Libraries: PIL, matplotlib, numpy, torch

FILE: ASSIGNMENT_6.ipynb
Libraries: PIL, matplotlib, requests, torch, torchvision

FILE: Untitled11.ipynb
Models: LogisticRegression
Libraries: matplotlib, numpy, sklearn, tensorflow

FILE: Untitled13.ipynb
Models: gpt2
Libraries: pandas, transformers


In [47]:
%%writefile /content/app.py

import os
import json
import re
import shutil
import streamlit as st
import torch

from git import Repo
from transformers import AutoTokenizer, AutoModelForCausalLM


MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"


# --------------------------------------------------
# LOAD QWEN MODEL
# --------------------------------------------------

@st.cache_resource
def load_model():

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=torch.float16,
        device_map="auto"
    )

    return tokenizer, model


# --------------------------------------------------
# CLONE GITHUB REPOSITORY
# --------------------------------------------------

def clone_repository(github_url):

    repo_folder = "/content/github_repo"

    if os.path.exists(repo_folder):
        shutil.rmtree(repo_folder)

    Repo.clone_from(github_url, repo_folder)

    return repo_folder


# --------------------------------------------------
# EXTRACT NOTEBOOK CODE
# --------------------------------------------------

def extract_repository_code(repo_folder):

    notebooks = []

    for root, dirs, files in os.walk(repo_folder):

        dirs[:] = [
            d for d in dirs
            if d not in [".git", "__pycache__", "venv", ".venv"]
        ]

        for file in files:

            if not file.endswith(".ipynb"):
                continue

            path = os.path.join(root, file)

            try:

                with open(path, "r", encoding="utf-8") as f:
                    notebook = json.load(f)

                code = ""

                for cell in notebook.get("cells", []):

                    if cell.get("cell_type") == "code":

                        code += "".join(
                            cell.get("source", [])
                        ) + "\n"

                notebooks.append({
                    "filename": file,
                    "code": code
                })

            except Exception:
                pass

    return notebooks


# --------------------------------------------------
# DETECT FACTS FROM CODE
# --------------------------------------------------

def detect_facts(code):

    patterns = {

        "Datasets": [
            "FashionMNIST",
            "CIFAR10",
            "CIFAR100",
            "ImageNet",
            "MNIST"
        ],

        "Models": [
            "Generator",
            "Discriminator",
            "VGG19",
            "LogisticRegression",
            "gpt2",
            "GPT2"
        ],

        "Libraries": [
            "torch",
            "torchvision",
            "tensorflow",
            "transformers",
            "numpy",
            "matplotlib",
            "PIL",
            "sklearn",
            "requests",
            "pandas"
        ]
    }

    facts = {}

    for category, items in patterns.items():

        found = []

        for item in items:

            if re.search(
                r"\b" + re.escape(item) + r"\b",
                code
            ):

                found.append(item)

        if found:
            facts[category] = sorted(set(found))

    return facts


# --------------------------------------------------
# GENERATE EXPLANATION
# --------------------------------------------------

def generate_file_explanation(
    filename,
    code,
    facts,
    tokenizer,
    model
):

    code = code[:7000]

    fact_text = json.dumps(
        facts,
        indent=2
    )

    prompt = f"""
You are a college project code explainer.

Analyze ONLY the provided notebook information.

Do not invent anything.

Notebook:
{filename}

Detected facts:
{fact_text}

Code:
{code}

Write a short explanation using exactly these sections:

### Purpose
Explain what this notebook does.

### Technologies
Mention only the detected libraries.

### Dataset
Mention only a dataset clearly visible in the code.
If none is detected, say:
Not clearly identified.

### Model or Algorithm
Mention only models or algorithms clearly visible in the code.

### Process
Explain the main steps visible in the code.

### Output
Explain what the code produces or displays.

Keep the explanation simple and factual.
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=8192
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False
        )

    generated = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    )


# --------------------------------------------------
# STREAMLIT INTERFACE
# --------------------------------------------------

st.set_page_config(
    page_title="GitHub Repository Code Explainer",
    page_icon="💻"
)

st.title("💻 GitHub Repository Code Explainer")

st.write(
    "Enter a public GitHub repository URL to "
    "analyze and explain its source code."
)

github_url = st.text_input(
    "GitHub Repository URL",
    placeholder="https://github.com/username/repository"
)


if st.button("Explain Repository"):

    if not github_url:

        st.warning(
            "Please enter a GitHub repository URL."
        )

    else:

        try:

            # Clone
            with st.spinner(
                "Cloning repository..."
            ):

                repo_folder = clone_repository(
                    github_url
                )

            st.success(
                "Repository cloned successfully!"
            )


            # Extract
            with st.spinner(
                "Extracting notebook code..."
            ):

                notebooks = extract_repository_code(
                    repo_folder
                )

            if not notebooks:

                st.error(
                    "No Jupyter Notebook files found."
                )

            else:

                st.info(
                    f"Found {len(notebooks)} notebook(s)."
                )


                # Load model
                with st.spinner(
                    "Loading Qwen model..."
                ):

                    tokenizer, model = load_model()


                st.subheader(
                    "📖 Repository Explanation"
                )


                # Analyze each notebook
                for notebook in notebooks:

                    filename = notebook["filename"]
                    code = notebook["code"]

                    facts = detect_facts(code)

                    with st.expander(
                        f"📄 {filename}",
                        expanded=True
                    ):

                        st.write(
                            "Detected information:"
                        )

                        if facts:

                            for category, values in facts.items():

                                st.write(
                                    f"**{category}:** "
                                    + ", ".join(values)
                                )

                        else:

                            st.write(
                                "No specific facts detected."
                            )


                        with st.spinner(
                            f"Analyzing {filename}..."
                        ):

                            explanation = (
                                generate_file_explanation(
                                    filename,
                                    code,
                                    facts,
                                    tokenizer,
                                    model
                                )
                            )

                        st.markdown(
                            explanation
                        )


        except Exception as e:

            st.error(
                f"Error: {str(e)}"
            )

Overwriting /content/app.py


In [48]:
import py_compile

py_compile.compile(
    "/content/app.py",
    doraise=True
)

print("app.py syntax is correct!")

app.py syntax is correct!


In [49]:
%%writefile /content/backend.py

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

import os
import json
import re
import shutil

from git import Repo

from app import load_model, generate_file_explanation


app = FastAPI(
    title="GitHub Repository Code Explainer API",
    description="API for analyzing and explaining GitHub repository code",
    version="1.0"
)


class RepositoryRequest(BaseModel):
    github_url: str


# --------------------------------------------------
# CLONE REPOSITORY
# --------------------------------------------------

def clone_repository(github_url):

    repo_folder = "/content/api_github_repo"

    if os.path.exists(repo_folder):
        shutil.rmtree(repo_folder)

    Repo.clone_from(
        github_url,
        repo_folder
    )

    return repo_folder


# --------------------------------------------------
# EXTRACT NOTEBOOKS
# --------------------------------------------------

def extract_repository_code(repo_folder):

    notebooks = []

    for root, dirs, files in os.walk(repo_folder):

        dirs[:] = [
            d for d in dirs
            if d not in [
                ".git",
                "__pycache__",
                "venv",
                ".venv"
            ]
        ]

        for file in files:

            if not file.endswith(".ipynb"):
                continue

            path = os.path.join(
                root,
                file
            )

            try:

                with open(
                    path,
                    "r",
                    encoding="utf-8"
                ) as f:

                    notebook = json.load(f)

                code = ""

                for cell in notebook.get(
                    "cells",
                    []
                ):

                    if cell.get(
                        "cell_type"
                    ) == "code":

                        code += "".join(
                            cell.get(
                                "source",
                                []
                            )
                        ) + "\n"

                notebooks.append({
                    "filename": file,
                    "code": code
                })

            except Exception:
                pass

    return notebooks


# --------------------------------------------------
# DETECT FACTS
# --------------------------------------------------

def detect_facts(code):

    patterns = {

        "Datasets": [
            "FashionMNIST",
            "CIFAR10",
            "CIFAR100",
            "ImageNet",
            "MNIST"
        ],

        "Models": [
            "Generator",
            "Discriminator",
            "VGG19",
            "LogisticRegression",
            "gpt2",
            "GPT2"
        ],

        "Libraries": [
            "torch",
            "torchvision",
            "tensorflow",
            "transformers",
            "numpy",
            "matplotlib",
            "PIL",
            "sklearn",
            "requests",
            "pandas"
        ]
    }

    facts = {}

    for category, items in patterns.items():

        found = []

        for item in items:

            if re.search(
                r"\b" + re.escape(item) + r"\b",
                code
            ):

                found.append(item)

        if found:

            facts[category] = sorted(
                set(found)
            )

    return facts


# --------------------------------------------------
# API HOME
# --------------------------------------------------

@app.get("/")
def home():

    return {
        "message":
        "GitHub Repository Code Explainer API is running"
    }


# --------------------------------------------------
# EXPLAIN REPOSITORY
# --------------------------------------------------

@app.post("/explain")
def explain_repository(
    request: RepositoryRequest
):

    try:

        # Clone repository
        repo_folder = clone_repository(
            request.github_url
        )


        # Extract notebooks
        notebooks = extract_repository_code(
            repo_folder
        )

        if not notebooks:

            raise HTTPException(
                status_code=400,
                detail="No Jupyter Notebook files found."
            )


        # Load model
        tokenizer, model = load_model()


        results = []


        # Analyze each notebook
        for notebook in notebooks:

            filename = notebook[
                "filename"
            ]

            code = notebook[
                "code"
            ]

            facts = detect_facts(
                code
            )


            explanation = (
                generate_file_explanation(
                    filename,
                    code,
                    facts,
                    tokenizer,
                    model
                )
            )


            results.append({

                "filename": filename,

                "detected_facts": facts,

                "explanation": explanation

            })


        return {

            "github_url":
            request.github_url,

            "files_analyzed":
            len(results),

            "results":
            results

        }


    except HTTPException:

        raise


    except Exception as e:

        raise HTTPException(
            status_code=500,
            detail=str(e)
        )

Overwriting /content/backend.py


In [50]:
import py_compile

py_compile.compile(
    "/content/backend.py",
    doraise=True
)

print("backend.py syntax is correct!")

backend.py syntax is correct!


In [51]:
!pkill -f "uvicorn backend:app" || true

^C


In [52]:
!uvicorn backend:app --host 0.0.0.0 --port 8000 > /content/fastapi.log 2>&1 &

In [54]:
print(open("/content/fastapi.log").read())


2026-10-05 17:15:33.134 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 17:15:33.134 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 17:15:33.481 
  command:

    streamlit run /usr/local/bin/uvicorn [ARGUMENTS]
2026-10-05 17:15:33.481 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 17:15:33.482 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 17:15:33.482 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 17:15:33.482 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 17:15:33.482 

In [55]:
import requests

response = requests.get("http://localhost:8000/")

print("Status:", response.status_code)
print("Response:", response.json())

Status: 200
Response: {'message': 'GitHub Repository Code Explainer API is running'}


In [56]:
import requests

response = requests.post(
    "http://localhost:8000/explain",
    json={
        "github_url": "https://github.com/Chaturyathukivakam/GenAI"
    }
)

print("Status:", response.status_code)

data = response.json()

print("Files analyzed:", data.get("files_analyzed"))

for result in data.get("results", []):
    print("\n" + "=" * 60)
    print("FILE:", result["filename"])
    print("=" * 60)

    print("\nDetected facts:")
    print(result["detected_facts"])

import requestsresponse = requests.post(    "http://localhost:8000/explain",    json={        "github_url": "https://github.com/Chaturyathukivakam/GenAI"    \})print("Status:", response.status_code)data = response.json()print("Files analyzed:", data.get("files_analyzed"))for result in data.get("results", []):    print("\\n" + "=" * 60)    print("FILE:", result["filename"])    print("=" * 60)    print("\\nDetected facts:")    print(result["detected_facts"])    print("\\nExplanation:")    print(result["explanation"])

 Status: 200
Files analyzed: 5

============================================================
FILE: ASSIGNMENT_4.ipynb
============================================================

Detected facts:
{'Datasets': ['FashionMNIST', 'MNIST'], 'Models': ['Discriminator', 'Generator'], 'Libraries': ['matplotlib', 'torch', 'torchvision']\}

Explanation:
### Purpose
The purpose of this notebook is to demonstrate how to create a generator and discriminator model using PyTorch, visualize the training process, and generate synthetic images from a given dataset.

### Technologies
- Libraries: `matplotlib`, `torch`, `torchvision`
- Not clearly identified

### Dataset
Not clearly identified

### Model or Algorithm
- **Generator**: A Convolutional Neural Network (CNN) that generates synthetic images based on input data.
- **Discriminator**: A Convolutional Neural Network (CNN) that identifies real vs. fake images.

### Process
1. **Install PyTorch**:
   ```bash
   !pip install torch torchvision matplotlib -q
   ```

2. **Import Libraries**:
   ```python
   import torch
   import torch.nn as nn
   import torch.optim as optim
   from torchvision import datasets, transforms
   from torchvision.utils import make_grid, save_image
   from torch.utils.data import DataLoader
   import matplotlib.pyplot as plt
   import os
   ```

3. **Create Folder to Save Generated Images**:
   ```python
   os.makedirs("generated_images", exist_ok=True)
   ```

4. **Load Fashion MNIST

============================================================
FILE: ASSIGNMENT 5 GEN AI .ipynb
============================================================

Detected facts:
{'Libraries': ['PIL', 'matplotlib', 'numpy', 'torch']\}

Explanation:
### Purpose
The purpose of this notebook is to demonstrate how to use the StyleGAN2-Ada-PyTorch library to generate images from pre-trained style transfer networks.

### Technologies
- Libraries: `PIL`, `matplotlib`, `numpy`, `torch`
- Not clearly identified

### Dataset
Not clearly identified

### Model or Algorithm
- The notebook uses the StyleGAN2-Ada-PyTorch library to load and generate images from pre-trained style transfer networks.

### Process
1. **Load Pre-trained Network**: The notebook loads a pre-trained style transfer network (`ffhq.pkl`) using the `legacy.load_network_pkl` function.
2. **Generate Images**: It generates two images using the generated latent vectors (`z₁` and `z₂`). These images are then displayed using Matplotlib for visualization.
3. **Modify Images**: The notebook allows changing dimensions of the generated images by modifying specific elements of the latent vector (`z`). This can be done through a simple modification of the latent vector values.
4. **Save Generated Images**: Finally, the modified images are saved as an animated GIF file named `stylegan2_interpolation.gif`.

### Output
The notebook outputs the generated images and their modifications,

============================================================
FILE: ASSIGNMENT_6.ipynb
============================================================

Detected facts:
{'Libraries': ['PIL', 'matplotlib', 'requests', 'torch', 'torchvision']\}

Explanation:
### Purpose
This notebook demonstrates how to perform Neural Style Transfer (NST) using PyTorch and torchvision libraries.

### Technologies
- Libraries:
  - `PIL` for image processing
  - `matplotlib` for plotting images
  - `torch` for deep learning operations
  - `torchvision` for image datasets and transformations
  - `requests` for making HTTP requests
  - `torchvision.transforms` for image transformations
  - `vgg19` from torchvision for creating a pre-trained model
  - `transforms` from torchvision.transforms for data augmentation
  - `pytorch-CycleGAN-and-pix2pix` repository containing CycleGAN model

### Dataset
Not clearly identified.

### Model or Algorithm
The notebook uses CycleGAN model from the `pytorch-CycleGAN-and-pix2pix` repository to perform NST.

### Process
1. **Load an Image**: The notebook loads an original image (`picasso.jpg`) and converts it into a tensor.
2. **Create a Style Image**: A random style image is generated using the `vgg19` model.
3. **Transfer the Style Image to the Original Image**: The notebook transfers the style image

============================================================
FILE: Untitled11.ipynb
============================================================

Detected facts:
{'Models': ['LogisticRegression'], 'Libraries': ['matplotlib', 'numpy', 'sklearn', 'tensorflow']\}

Explanation:
### Purpose
This notebook demonstrates how to train a logistic regression model on the MNIST dataset for handwritten digit classification.

### Technologies
- Libraries: `matplotlib`, `numpy`, `sklearn`, `tensorflow`

### Dataset
Not clearly identified

### Model or Algorithm
Logistic Regression

### Process
The notebook starts with importing necessary libraries and loading the MNIST dataset. It then normalizes and flattens the data before splitting it into training and testing sets. A logistic regression model is trained on the training set, and its performance is evaluated on the test set. Finally, generated images from the trained model are displayed.

### Output
The notebook outputs an image of a randomly selected digit from the MNIST dataset after training the logistic regression model.

============================================================
FILE: Untitled13.ipynb
============================================================

Detected facts:
{'Models': ['gpt2'], 'Libraries': ['pandas', 'transformers']\}

Explanation:
### Purpose
This notebook demonstrates various text generation tasks using the GPT-2 model from the Hugging Face Transformers library.

### Technologies
- Libraries: `pandas` for data manipulation and `transformers` for natural language processing (NLP) tasks like text generation.

### Dataset
Not clearly identified.

### Model or Algorithm
GPT-2

### Process
The notebook contains several scripts that demonstrate different NLP tasks using the GPT-2 model. These include generating text based on prompts, sampling text with specific parameters, and performing text completion tasks.

#### Script 1: Text Generation
```python
# Install transformers if not already installed
!pip install transformers

from transformers import pipeline, set_seed

generator = pipeline("text-generation", model="gpt2")
prompt = "The old house at the end of the street"

print(prompt)
set_seed(10)

output1 = generator(
    prompt,
    max_new_tokens=40,
    temperature=0.3,
    do_sample=True
)

print(output1[0]["generated_text"])
```

#### Script 2: Sampling Text
```python
# Install transformers if not already installed
!pip install transformers

from transformers import pipeline$0    print("\nExplanation:")
    print(result["explanation"])

Status: 200
Files analyzed: 5

FILE: ASSIGNMENT_4.ipynb

Detected facts:
{'Datasets': ['FashionMNIST', 'MNIST'], 'Models': ['Discriminator', 'Generator'], 'Libraries': ['matplotlib', 'torch', 'torchvision']}

Explanation:
### Purpose
The purpose of this notebook is to demonstrate how to create a generator and discriminator model using PyTorch, visualize the training process, and generate synthetic images from a given dataset.

### Technologies
- Libraries: `matplotlib`, `torch`, `torchvision`
- Not clearly identified

### Dataset
Not clearly identified

### Model or Algorithm
- **Generator**: A Convolutional Neural Network (CNN) that generates synthetic images based on input data.
- **Discriminator**: A Convolutional Neural Network (CNN) that identifies real vs. fake images.

### Process
1. **Install PyTorch**:
   ```bash
   !pip install torch torchvision matplotlib -q
   ```

2. **Import Libraries**:
   ```python
   import torch
   import torch.nn as nn
   import torch.optim as optim
